# The ML Project Loop — from scratch

> Lesson: [The ML Project Loop](https://ml-viz-ruby.vercel.app/wiki/ml-project-loop)
> · Copy to Drive to run and edit.

This notebook walks a **single problem — daily demand forecasting — all the way
around the six-stage loop**:

`data → hypothesis space → objective → optimization → evaluation → deployment feedback`

Everything is plain NumPy so you can see each slot with no framework in the way.
We finish with a **slot-placement drill**: given techniques you may not have been
taught, name which slot each one changes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style to match the site.
plt.rcParams.update({
    "figure.facecolor": "#0f1117", "axes.facecolor": "#0f1117",
    "savefig.facecolor": "#0f1117", "text.color": "#e2e8f0",
    "axes.labelcolor": "#e2e8f0", "xtick.color": "#94a3b8",
    "ytick.color": "#94a3b8", "axes.edgecolor": "#2e3347",
    "grid.color": "#2e3347", "axes.grid": True, "figure.figsize": (9, 3.4),
})
BRAND, TEAL, ROSE, YELLOW = "#6366f1", "#14b8a6", "#f43f5e", "#eab308"
rng = np.random.default_rng(7)  # seed = 7, the canonical dataset

## The canonical dataset: `daily-demand` (seed = 7)

The exact generative process from the wiki page — 730 days with trend, weekly
seasonality, occasional promotions, and a permanent **regime shift at day 400**
(a new store opens). That shift is the whole point of the *feedback* stage.

$$y_t = (200 + 0.15\,t)\cdot w_{(t\bmod 7)}\cdot p_t + 60\cdot\mathbb{1}[t\ge 400] + \varepsilon_t$$

In [ ]:
T = 730
t = np.arange(T)
w = np.array([1.00, 0.95, 0.98, 1.00, 1.15, 1.30, 1.20])   # Mon..Sun
weekday = t % 7

trend = 200 + 0.15 * t
weekly = w[weekday]

promo_days = rng.random(T) < 0.04                # ~4% of days
promo = np.where(promo_days, rng.choice([1.6, 1.9], size=T), 1.0)

regime = 60.0 * (t >= 400)                        # permanent level jump
noise = rng.normal(0, 12, size=T)

y = trend * weekly * promo + regime + noise
print(f"generated {T} days, mean demand {y.mean():.1f}, "
      f"pre-shift mean {y[:400].mean():.1f}, post-shift mean {y[400:].mean():.1f}")

### Stage 3 preview — see the raw material

Before modelling, *look* at the data. The weekly ripple, the promo spikes, and the
step up at day 400 are all visible — and each will stress a different slot.

In [ ]:
fig, ax = plt.subplots()
ax.plot(t, y, color=BRAND, lw=0.9, label="demand")
ax.scatter(t[promo_days], y[promo_days], color=YELLOW, s=14, zorder=3, label="promo day")
ax.axvline(400, color=ROSE, ls="--", lw=1.5, label="regime shift (day 400)")
ax.set_xlabel("day"); ax.set_ylabel("units"); ax.legend(loc="upper left", fontsize=8)
ax.set_title("daily-demand", color="#e2e8f0"); plt.tight_layout(); plt.show()

**What to notice.** The series is non-stationary in two ways the model must
survive: a slow trend (fine — lags absorb it) and a *sudden* level shift at 400
(not fine — nothing in the features predicts it). Hold that thought for stage 6.

## 1 · Data — features that respect time

We predict $y_t$ from things known *before* $t$: the previous day $y_{t-1}$, the
same weekday last week $y_{t-7}$, and calendar dummies for day-of-week. We start at
$t=7$ (need the 7-day lag), and we split **chronologically** — a random shuffle
would leak the future into training.

In [ ]:
def make_features(y):
    T = len(y)
    rows, targets, idx = [], [], []
    for i in range(7, T):
        dow = np.zeros(7); dow[i % 7] = 1.0
        rows.append(np.concatenate(([1.0, y[i-1], y[i-7]], dow[1:])))  # drop 1 dummy
        targets.append(y[i]); idx.append(i)
    return np.array(rows), np.array(targets), np.array(idx)

X, target, idx = make_features(y)
# Chronological split by original day index.
tr = idx < 600
va = (idx >= 600) & (idx < 665)
te = idx >= 665
print(f"features: {X.shape[1]} cols | train {tr.sum()}  val {va.sum()}  test {te.sum()}")

## 2 · Hypothesis space — start with the tabular baseline

A **linear model** on those features: $\hat y = X\beta$. That is already an
inductive bias — additive, no interactions. Widening to gradient boosting or an RNN
later is *only* a change to this slot; the other five stay put.

## 3 · Objective — what "good" means

Squared error with an $\ell_2$ (ridge) penalty so the lag weights can't explode:

$$J(\beta) = \lVert X\beta - y\rVert^2 + \lambda\lVert\beta\rVert^2$$

Swapping in MAE or the pinball (quantile) loss would encode a different business
cost — a different objective, same data and model.

## 4 · Optimization — search for the best $\beta$

Ridge regression is convex with a closed form (the normal equations); anything
bigger would be gradient descent. The objective's *shape* decides how hard this slot
is — here it's a single matrix solve.

In [ ]:
def ridge_fit(X, y, lam=1.0):
    n = X.shape[1]
    R = lam * np.eye(n); R[0, 0] = 0.0          # don't penalise the intercept
    return np.linalg.solve(X.T @ X + R, X.T @ y)  # (X'X + λI)^{-1} X'y

# Pick λ on the validation slot (a tiny sweep — model selection is stage 5 in miniature).
best = min([0.0, 0.1, 1.0, 10.0, 100.0],
           key=lambda lam: np.mean((X[va] @ ridge_fit(X[tr], target[tr], lam) - target[va])**2))
beta = ridge_fit(X[tr], target[tr], best)
print(f"chosen λ = {best} | intercept {beta[0]:.1f}, "
      f"lag1 weight {beta[1]:.3f}, lag7 weight {beta[2]:.3f}")

## 5 · Evaluation — honestly, on time-ordered data

We score one-step-ahead predictions on the **held-out future** (test slot). Report
MAE and MAPE. Because the split is chronological, this is a real
[walk-forward](https://ml-viz-ruby.vercel.app/wiki/walk-forward-validation)
estimate — no leakage.

In [ ]:
def mae(a, b):  return np.mean(np.abs(a - b))
def mape(a, b): return np.mean(np.abs((a - b) / a)) * 100

pred_te = X[te] @ beta
print(f"test MAE  {mae(target[te], pred_te):6.2f} units")
print(f"test MAPE {mape(target[te], pred_te):6.2f} %")

fig, ax = plt.subplots()
ax.plot(idx[te], target[te], color=TEAL, lw=1.2, label="actual")
ax.plot(idx[te], pred_te, color=BRAND, lw=1.2, ls="--", label="predicted")
ax.set_xlabel("day"); ax.set_ylabel("units"); ax.legend(fontsize=8)
ax.set_title("stage 5 — one-step-ahead on the held-out future", color="#e2e8f0")
plt.tight_layout(); plt.show()

**What to notice.** The lag features make this easy *because the test slot is all
post-shift* and the model was trained on post-shift data too. The next stage breaks
exactly that assumption.

## 6 · Deployment feedback — the loop closes (or the model rots)

Imagine the model was trained **before** the new store opened — only on days
`[7, 400)` — and then served straight through the regime shift. Watch the residuals:
a drift monitor watching rolling error should fire, triggering a **retrain on fresh
data** that feeds back into stage 1.

In [ ]:
beta_pre = ridge_fit(X[idx < 400], target[idx < 400], best)
resid = target - X @ beta_pre
roll = np.convolve(np.abs(resid), np.ones(14)/14, mode="valid")  # 14-day rolling MAE
roll_x = idx[13:]

fig, ax = plt.subplots()
ax.plot(roll_x, roll, color=YELLOW, lw=1.3, label="14-day rolling MAE")
ax.axvline(400, color=ROSE, ls="--", lw=1.5, label="regime shift")
ax.axhline(np.abs(resid[idx < 400]).mean(), color=TEAL, lw=1.0, ls=":", label="pre-shift error")
ax.set_xlabel("day"); ax.set_ylabel("abs error"); ax.legend(fontsize=8)
ax.set_title("stage 6 — drift: a stale model rots after day 400", color="#e2e8f0")
plt.tight_layout(); plt.show()

pre = np.abs(resid[idx < 400]).mean(); post = np.abs(resid[idx >= 400]).mean()
print(f"error before shift {pre:5.2f}  →  after shift {post:5.2f}  "
      f"({post/pre:.1f}x worse) — this is the retrain trigger")

**What to notice.** Nothing about the *model* changed — the world did. No amount
of better optimization or a fancier hypothesis space fixes a data/feedback problem;
only closing the loop (detect → retrain → redeploy) does. That's why production ML
is a loop, not a pipeline.

## ✏️ Your turn — the slot-placement drill

The real skill is placing a technique you were **never taught**. For each one below,
decide which single slot it primarily modifies:

`data`, `hypothesis-space`, `objective`, `optimization`, `evaluation`, `feedback`.

Fill in the dictionary, then run the check cell — it stays silent when you're right.

In [ ]:
# TODO(you): map each technique to the slot it primarily changes.
placements = {
    "batch normalization":     "...",   # stabilises training of deep nets
    "SMOTE oversampling":      "...",   # synthesises minority-class examples
    "label smoothing":         "...",   # softens one-hot targets
    "early stopping":          "...",   # halts training via a validation metric
    "temperature scaling":     "...",   # post-hoc fix so probabilities are calibrated
    "shadow deployment":       "...",   # run the new model silently on live traffic
}

In [ ]:
solution = {
    "batch normalization":     "optimization",       # smooths the loss surface -> easier search
    "SMOTE oversampling":      "data",               # changes the training distribution
    "label smoothing":         "objective",          # changes the loss / target
    "early stopping":          "optimization",        # where in the search you stop (capacity control via optimization)
    "temperature scaling":     "evaluation",          # calibration, measured & fixed at eval time
    "shadow deployment":       "feedback",            # a production-feedback safety mechanism
}
for k, v in placements.items():
    assert v == solution[k], f"{k!r}: think again — which slot was breaking?"
print("all placements correct — you can read techniques as slot-changes now.")

<details>
<summary>Solution & reasoning</summary>

| Technique | Slot | Why |
|---|---|---|
| batch normalization | `optimization` | it reconditions the loss surface so gradient descent converges — not a new function family |
| SMOTE oversampling | `data` | it edits the training distribution, nothing else |
| label smoothing | `objective` | it changes the target/loss to curb over-confidence |
| early stopping | `optimization` | it's a choice about *when to stop the search* (a capacity control acting through optimization) |
| temperature scaling | `evaluation` | calibration is diagnosed and corrected at evaluation time |
| shadow deployment | `feedback` | it's a production safety valve in the deploy→monitor→learn loop |

Reasonable people put `early stopping` under evaluation (it *reads* a validation
metric) — the point of the drill is the argument, not a single "right" cell.
</details>

## Key takeaways

- Every technique is a change to **one or two slots** — name the slot and the failure
  it fixes, and you can place things you've never seen.
- One problem (**daily demand**) exercised all six stages; the same problem recurs
  across the site's Time Series, Streaming ML, and Bayesian Methods courses.
- Production ML is a **loop**: the regime shift was a *data/feedback* failure that no
  amount of modelling could fix — only retraining could.